# PHLOP Results Analysis

Compare predictions across all models (SmolVLM, InternVL3, VideoLLaMA3) for **static camera, no physics** configuration.
Plots per-question-type histograms showing correct vs total counts.

In [ ]:
import os, sys, json, glob
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path(".").resolve()))
from phlop_eval_common import answer_accuracy

RESULTS_DIR = "results"
CAMERA = "static"
PHYSICS = "no_physics"

## Load Predictions

Scan for zero-shot and fine-tuned prediction JSON files matching static / no_physics.

In [ ]:
def load_predictions(results_dir, camera, physics_tag):
    """Load all prediction JSONs matching the given camera/physics config.
    Returns a dict: model_label -> list[dict]."""
    all_preds = {}

    patterns = [
        (f"{results_dir}/*_zero_shot_predictions_test_{camera}_{physics_tag}.json", "zero_shot"),
        (f"{results_dir}/*_finetuned_predictions_*_test_{camera}_{physics_tag}.json", "finetuned"),
    ]

    for pattern, phase in patterns:
        for fpath in sorted(glob.glob(pattern)):
            fname = os.path.basename(fpath)
            parts = fname.replace(".json", "").split("_")

            if phase == "zero_shot":
                model_prefix = parts[0]
                label = f"{model_prefix} (zero-shot)"
            else:
                model_prefix = parts[0]
                pred_idx = fname.index("_finetuned_predictions_") + len("_finetuned_predictions_")
                rest = fname[pred_idx:].replace(".json", "")
                rest_parts = rest.split("_")
                config_parts = rest_parts[:-3]  # drop test_{camera}_{physics}
                config_name = "_".join(config_parts)
                label = f"{model_prefix} ({config_name})"

            with open(fpath) as f:
                preds = json.load(f)
            all_preds[label] = preds
            print(f"  Loaded {len(preds):>5} predictions: {label} <- {fname}")

    return all_preds


all_preds = load_predictions(RESULTS_DIR, CAMERA, PHYSICS)
print(f"\nTotal: {len(all_preds)} model configurations loaded")

## Per-Question-Type Statistics

In [ ]:
def compute_per_type_stats(preds):
    """Return {question_type: {correct, total, accuracy}} for a list of predictions."""
    by_type = defaultdict(lambda: {"correct": 0, "total": 0})
    for r in preds:
        qt = r.get("question_type") or r.get("category") or "unknown"
        pred = r.get("prediction", "")
        target = r.get("target", r.get("true_answer", ""))
        by_type[qt]["total"] += 1
        by_type[qt]["correct"] += answer_accuracy(pred, target)
    for qt in by_type:
        t = by_type[qt]["total"]
        by_type[qt]["accuracy"] = by_type[qt]["correct"] / t if t > 0 else 0.0
    return dict(by_type)


model_stats = {}
for label, preds in all_preds.items():
    stats = compute_per_type_stats(preds)
    model_stats[label] = stats
    total_correct = sum(v["correct"] for v in stats.values())
    total_all = sum(v["total"] for v in stats.values())
    acc = total_correct / total_all if total_all else 0
    print(f"{label}: {total_correct}/{total_all} correct ({acc:.2%})")
    for qt in sorted(stats, key=lambda k: -stats[k]["total"]):
        s = stats[qt]
        print(f"  {qt:>30}: {s['correct']:>4}/{s['total']:>4} ({s['accuracy']:.1%})")

## Correct vs Total by Question Type (per model)

In [ ]:
all_qtypes = sorted(
    set(qt for stats in model_stats.values() for qt in stats),
    key=lambda qt: -max(s.get(qt, {}).get("total", 0) for s in model_stats.values()),
)

n_models = len(model_stats)
if n_models == 0:
    print("No predictions found.")
else:
    cols = min(3, n_models)
    rows = (n_models + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows), squeeze=False)

    for idx, (label, stats) in enumerate(model_stats.items()):
        ax = axes[idx // cols][idx % cols]
        totals = [stats.get(qt, {}).get("total", 0) for qt in all_qtypes]
        corrects = [stats.get(qt, {}).get("correct", 0) for qt in all_qtypes]

        y = np.arange(len(all_qtypes))
        ax.barh(y, totals, color="#d4d4d4", label="Total")
        ax.barh(y, corrects, color="#4c9f70", label="Correct")

        ax.set_yticks(y)
        ax.set_yticklabels(all_qtypes, fontsize=8)
        ax.set_xlabel("Count")
        ax.set_title(label, fontsize=10)
        ax.legend(loc="lower right", fontsize=8)
        ax.invert_yaxis()

        for i, (t, c) in enumerate(zip(totals, corrects)):
            if t > 0:
                ax.text(t + max(totals) * 0.01, i, f"{c}/{t} ({c/t:.0%})",
                        va="center", fontsize=7)

    for idx in range(n_models, rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)

    plt.suptitle(f"Correct vs Total by Question Type ({CAMERA}, {PHYSICS})",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    chart_path = os.path.join(RESULTS_DIR, "chart_per_model_qtype_counts.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart_path}")

## Grouped Accuracy Comparison Across Models

In [ ]:
if not model_stats:
    print("No predictions found.")
else:
    model_labels = list(model_stats.keys())
    n_types = len(all_qtypes)
    n_m = len(model_labels)
    bar_h = 0.8 / n_m
    colors = plt.cm.tab10.colors

    fig, ax = plt.subplots(figsize=(12, max(5, n_types * 0.6)))
    y = np.arange(n_types)

    for i, label in enumerate(model_labels):
        stats = model_stats[label]
        accs = [stats.get(qt, {}).get("accuracy", 0) for qt in all_qtypes]
        counts = [stats.get(qt, {}).get("total", 0) for qt in all_qtypes]
        offsets = y + i * bar_h - (n_m - 1) * bar_h / 2
        bars = ax.barh(offsets, accs, height=bar_h, label=label,
                       color=colors[i % len(colors)])
        for bar, c in zip(bars, counts):
            if c > 0:
                ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                        f"n={c}", va="center", fontsize=6)

    ax.set_yticks(y)
    ax.set_yticklabels(all_qtypes, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Accuracy by Question Type — All Models ({CAMERA}, {PHYSICS})")
    ax.legend(loc="lower right", fontsize=7, ncol=1)
    plt.tight_layout()
    chart_path = os.path.join(RESULTS_DIR, "chart_grouped_accuracy_all_models.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart_path}")

## Overall Accuracy Summary Table

In [ ]:
if not model_stats:
    print("No predictions found.")
else:
    print(f"{'Model':<45} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
    print("-" * 75)

    summary_rows = []
    for label, stats in model_stats.items():
        total_c = sum(v["correct"] for v in stats.values())
        total_n = sum(v["total"] for v in stats.values())
        acc = total_c / total_n if total_n else 0
        summary_rows.append((label, total_c, total_n, acc))

    summary_rows.sort(key=lambda r: -r[3])
    for label, c, n, acc in summary_rows:
        print(f"{label:<45} {c:>8} {n:>8} {acc:>10.2%}")

    fig, ax = plt.subplots(figsize=(10, max(3, len(summary_rows) * 0.5)))
    labels = [r[0] for r in summary_rows]
    accs = [r[3] for r in summary_rows]
    y = np.arange(len(labels))
    bars = ax.barh(y, accs, color=[plt.cm.tab10.colors[i % 10] for i in range(len(labels))])
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Answer Accuracy")
    ax.set_title(f"Overall Accuracy — All Models ({CAMERA}, {PHYSICS})")
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{acc:.2%}", va="center", fontsize=8)
    ax.invert_yaxis()
    plt.tight_layout()
    chart_path = os.path.join(RESULTS_DIR, "chart_overall_accuracy_all_models.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart_path}")